In [1]:
import os
from dotenv import load_dotenv

In [2]:
from langchain_deepseek import ChatDeepSeek
from langchain_core import tools
from langchain.agents import create_agent
from langchain_tavily import TavilySearch, TavilyExtract
from tavily import TavilyClient
from langchain.tools import tool
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.agents.middleware import ModelCallLimitMiddleware

In [3]:
from libmumd import convert_file

In [4]:
load_dotenv()

True

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import math

class ScientificGraph:
    """
    A production-grade class for generating scientific plots with linear regression,
    error bars, automatic significant-figure formatting, and scientific tick notation.
    """
    def __init__(self, x, y, x_label, y_label, title="", x_err=None, y_err=None, x_unit="", y_unit=""):
        self.x = np.array(x, dtype=float)
        self.y = np.array(y, dtype=float)
        self.x_label = x_label
        self.y_label = y_label
        self.x_unit = f" ({x_unit})" if x_unit else ""
        self.y_unit = f" ({y_unit})" if y_unit else ""
        self.title = title
        self.x_err = np.array(x_err, dtype=float) if x_err is not None else None
        self.y_err = np.array(y_err, dtype=float) if y_err is not None else None
        
        # Perform linear regression
        self.slope, self.intercept, self.r_value, self.p_value, self.std_err = linregress(self.x, self.y)
        self.r_squared = self.r_value ** 2
        
        # Calculate uncertainties of slope and intercept
        n = len(self.x)
        self.slope_err = self.std_err
        self.intercept_err = self.std_err * np.sqrt(np.sum(self.x**2) / n)

    @staticmethod
    def format_val_err(val, err, sig_figs=2):
        """
        Formats a value and its error according to standard scientific rules:
        1. Error is rounded to sig_figs (default 2).
        2. Value is rounded to the same decimal place as the error.
        """
        if err == 0 or np.isnan(err):
            return f"{val:.{sig_figs}e}", f"{0:.{sig_figs}e}"
        
        # Determine the exponent of the error to figure out the decimal places
        # e.g., 0.0124 -> exp = -2 -> decimal_places = 2
        exponent = int(math.floor(math.log10(abs(err))))
        # We want `sig_figs` in the mantissa. If sig_figs=1, and err=0.0124, we want 0.01
        # So we shift: decimal_places = sig_figs - exponent - 1
        decimal_places = sig_figs - exponent - 1
        
        # Round both using the calculated decimal places
        val_round = round(val, decimal_places)
        err_round = round(err, decimal_places)
        
        # If the number is very small or very large, fall back to scientific notation
        if abs(val_round) >= 1e4 or (0 < abs(val_round) < 1e-3):
            # Format as scientific notation with the same number of decimal places
            val_str = f"{val_round:.{decimal_places}e}"
            err_str = f"{err_round:.{decimal_places}e}"
            # Strip trailing zeros in mantissa for cleanliness
            val_str = val_str.replace("e+0", "e").replace("e-0", "e-")
            err_str = err_str.replace("e+0", "e").replace("e-0", "e-")
            return val_str, err_str
        
        # Otherwise, standard decimal notation
        return f"{val_round:.{decimal_places}f}", f"{err_round:.{decimal_places}f}"

    def plot(self, figsize=(9, 6), color='#1f77b4', fit_color='#d62728', show_eq=True):
        """
        Generates the plot with error bars, fit line, scientific ticks, and LaTeX labels.
        """
        # Use a clean, journal-style style
        plt.style.use('default')
        fig, ax = plt.subplots(figsize=figsize, dpi=150)
        
        # Plot data points with error bars if provided
        ax.errorbar(self.x, self.y, xerr=self.x_err, yerr=self.y_err, 
                    fmt='o', color=color, capsize=4, label='داده‌های تجربی', 
                    ecolor='black', elinewidth=1, markeredgecolor='black', zorder=3)
        
        # Generate fit line data
        x_fit = np.linspace(min(self.x) * 0.95, max(self.x) * 1.05, 100)
        y_fit = self.slope * x_fit + self.intercept
        ax.plot(x_fit, y_fit, color=fit_color, linestyle='-', linewidth=2, label='برازش خطی', zorder=2)
        
        # Format the equation and statistics
        if show_eq:
            slope_str, slope_err_str = self.format_val_err(self.slope, self.slope_err)
            int_str, int_err_str = self.format_val_err(self.intercept, self.intercept_err)
            
            eq_text = (f"$y = ({slope_str} \\pm {slope_err_str})x + ({int_str} \\pm {int_err_str})$\n"
                       f"$R^2 = {self.r_squared:.4f}$")
            
            # Place text box in the top-left corner
            ax.text(0.05, 0.95, eq_text, transform=ax.transAxes, fontsize=12,
                    verticalalignment='top', 
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray'))

        # Labels and title
        ax.set_xlabel(f"{self.x_label}{self.x_unit}", fontsize=14, labelpad=10)
        ax.set_ylabel(f"{self.y_label}{self.y_unit}", fontsize=14, labelpad=10)
        if self.title:
            ax.set_title(self.title, fontsize=16, fontweight='bold', pad=15)
            
        # Scientific Ticks configuration
        ax.tick_params(axis='both', which='major', labelsize=12, direction='in', length=6, width=1.5)
        ax.tick_params(axis='both', which='minor', direction='in', length=3, width=1)
        
        # Apply scientific notation formatter for ticks
        from matplotlib.ticker import ScalarFormatter, AutoMinorLocator
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_scientific(True)
        formatter.set_powerlimits((-3, 4)) # Use sci notation if < 1e-3 or >= 1e4
        ax.xaxis.set_major_formatter(formatter)
        ax.yaxis.set_major_formatter(formatter)
        
        # Add minor ticks for a more professional grid feel
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

        # Grid styling
        ax.grid(True, which='major', linestyle='--', alpha=0.7, zorder=1)
        ax.grid(True, which='minor', linestyle=':', alpha=0.4, zorder=1)
        
        # Legend
        ax.legend(loc='best', fontsize=12, framealpha=0.9, edgecolor='gray')
        
        plt.tight_layout()
        plt.show()
        
        return self.slope, self.intercept, self.slope_err, self.intercept_err

    def get_results(self):
        """
        Returns a dictionary of the fit results for easy programmatic use.
        """
        return {
            "slope": self.slope,
            "slope_err": self.slope_err,
            "intercept": self.intercept,
            "intercept_err": self.intercept_err,
            "r_squared": self.r_squared
        }

In [6]:
instructions = """
You are writing a laboratory report in Persian, formatted as Obsidian Markdown.
The report MUST contain all of the following 15 sections, in this exact order.
Each section header must be written in Persian, as shown in parentheses.
Any section whose content requires an image, circuit diagram, graph, or
screenshot MUST be marked with a `==== TODO: ... ====` placeholder so the
user can insert it manually afterward.

1. Experiment Title
   (عنوان آزمایش)
   - A short, descriptive title of the experiment.

2. Date of Experiment
   (تاریخ انجام آزمایش)
   - The date the experiment was performed.

3. Date of Report Submission
   (تاریخ تحویل گزارشکار)
   - The date the report is being submitted.

4. Author's Full Name
   (نام و نام خانوادگی)
   - The student's first and last name.

5. Lab Partner's Name
   (نام همکار)
   - The name of the collaborator / lab partner.

6. Abstract
   (چکیده آزمایش)
   - A single paragraph containing:
       * the objective of the experiment,
       * the measured parameters,
       * the method of analyzing the results,
       * the final conclusion.
   - Style: similar to the abstract of a scientific paper.
   - No bullet points — write it as one coherent paragraph.

7. Apparatus and Experimental Setup
   (وسایل آزمایش، توضیح مختصری از چیدمان آزمایش)
   - A list of the equipment used.
   - A brief description of how the setup was arranged.
   - If a photo or schematic of the setup is needed, emit:
       ==== TODO: insert photo/schematic of experimental setup ====

8. Theoretical Background
   (تئوری آزمایش)
   - The physical/mathematical theory behind the experiment.
   - Prefer sources OTHER than the lab manual; cite each reference.
   - Include equations in LaTeX where appropriate.

9. Experimental Procedure
   (روش انجام آزمایش)
   - A numbered, step-by-step description of what was actually done.

10. Circuit Diagram
    (رسم مدار)
    - If the experiment involves a circuit, emit:
        ==== TODO: insert circuit diagram ====
    - If no circuit is involved, write: "این آزمایش شامل مدار الکتریکی نیست."

11. Experimental Data and Sample Calculations
    (داده‌های آزمایش و نمونه محاسبات)
    - A table of the raw measured data.
    - At least one worked example calculation, step by step.
    - Follow the numerical-reporting rule below.

12. Graphs
    (نمودار به همراه شیب خط، عرض از مبدا و خطاهای مربوطه)
    - For every graph, emit a placeholder:
        ==== TODO: insert graph of <what> with fit line, slope, intercept, and error bars ====
    - Then state in text: the slope, the intercept, and the uncertainty of each.
    - Apply the numerical-reporting rule below.

13. Conclusion
    (نتیجه‌گیری و گزارش داده‌های اصلی به دست آمده به همراه خطاها و مقایسه با مقادیر نظری یا کتاب)
    - Report the main results WITH their uncertainties.
    - Compare against theoretical or textbook values.
    - Comment on sources of error and whether the result is consistent
      with the expected value within error.

14. Answers to Questions
    (پاسخ به سوالات)
    - Answer every question posed in the lab manual, one by one.

15. Suggestions for Improvement
    (پیشنهادات برای بهینه‌سازی آزمایش)
    - Concrete suggestions to reduce error or improve the experiment.

------------------------------------------------------------------
NUMERICAL REPORTING RULE (applies to sections 11, 12, and 13)
------------------------------------------------------------------
Whenever a measured value is reported with an uncertainty:
  * the uncertainty MUST be rounded to exactly ONE significant figure;
  * the measured value MUST be rounded to the SAME number of decimal
    places as the uncertainty;
  * always use the format:  value ± uncertainty  unit
  * example:  3.457 ± 0.0124  →  3.46 ± 0.01
"""

In [7]:
sample = """
This document serves as the theoretical background and procedural guide for the "Black Body Radiation" experiment. 
The goal is to investigate the Stefan-Boltzmann law and Lambert's laws regarding black body radiation.

**Apparatus:**
Thermal furnace (as a black body), precise digital thermometer, thermopile, diaphragm, precise voltmeter.

**Theoretical Background:**
A black body is an idealized physical body that completely absorbs all incident electromagnetic radiation, regardless of frequency. When heated, it also emits electromagnetic radiation across all frequencies. 
According to Wien's Displacement Law, the peak emission wavelength shifts with temperature: 
λ_max * T = C 
where T is absolute temperature, λ_max is the wavelength of peak emission, and C = 2.9 × 10^-3 m·K.

The Stefan-Boltzmann law states that the total radiant power emitted per unit surface area is proportional to the fourth power of the body's absolute temperature:
R(T) = σ T^4

To calculate the constant σ, we use the voltage V measured by the thermopile. The thermopile has a sensitivity of 16 mV / mW (meaning for every 1 mW of thermal energy it receives, it produces 16 mV of output). 
The relationship between the received power P and the measured voltage V is used with the geometric cross-section:
R(T) = (4 d^2 / (π l^2)) * P
(where d is the distance from the diaphragm to the thermopile, and l is the diameter of the diaphragm).

**Experimental Procedure:**
The experiment consists of three main parts:
1. **Measuring Voltage vs. Distance:** The furnace is kept at a constant high temperature (around 350°C). The thermopile is placed at various distances from the furnace, and the corresponding voltages are recorded to verify the inverse-square law.
2. **Measuring Voltage vs. Angle:** The distance is fixed (d = 20 cm) and the furnace temperature is fixed (around 350°C). The thermopile is rotated to vary the angle θ between the incident radiation and the thermopile surface. The voltage is recorded for angles from 0° to 50° (in steps of 10° then 5°). The goal is to plot V vs. cos(θ) to verify Lambert's cosine law.
3. **Verifying the Stefan-Boltzmann Law:** The furnace temperature is allowed to slowly decrease. For every 5 to 10 degrees drop in temperature, the thermopile voltage is recorded. A cardboard cover is used to prevent the thermopile from heating up prematurely. The results are plotted as R(T) vs. T^4.
"""

In [8]:
questions = """
1. What is the frequency range of the radiation emitted in this experiment?
2. How does a thermopile work?
3. Explain the failure of classical physics in justifying black body radiation (Ultraviolet Catastrophe).
"""

In [9]:
import pandas as pd
import numpy as np

# Dataset 1: Voltage vs Distance (from first image)
data_distance = pd.DataFrame({
    'd_cm': [20, 25, 30, 35, 40, 45, 50, 55],
    'inv_d2': [0.0025, 0.0016, 0.00111, 8.16327e-4, 6.25e-4, 4.93827e-4, 4.0e-4, 3.30579e-4],
    'T_C': [354, 353, 348, 355, 348, 352, 354, 346],
    'V_off_mV': [110, 90, 77, 82, 78, 71, 67, 63],
    'V_on_mV': [1412, 912, 690, 515, 386, 332, 255, 210],
    'V_net_mV': [1302, 822, 613, 433, 308, 261, 188, 147]
})

# Dataset 2: Voltage vs Temperature (from second image, 'k3 - sigma')
data_temp = pd.DataFrame({
    'T_C': [350, 339, 330, 319, 310, 300, 289, 280, 270],
    'T_K': [623, 612, 603, 592, 583, 573, 562, 553, 543],
    'T4_K4': [1.50644e11, 1.40283e11, 1.32212e11, 1.22825e11, 1.15525e11, 1.078e11, 9.97574e10, 9.35191e10, 8.69359e10],
    'V_off_mV': [25, 25, 21, 16, 23, 22, 23, 25, 24],
    'V_on_mV': [650, 605, 572, 535, 504, 474, 445, 412, 380],
    'V_net_mV': [625, 580, 551, 519, 481, 452, 422, 387, 356]
})

# Placeholder for the Angular Data (fill this when you have it)
# Format example: [0, 10, 20, 30, 40, 50]
angles_deg = []
volts_angular_mV = []

data_angle = pd.DataFrame({
    'theta_deg': angles_deg,
    'cos_theta': np.cos(np.radians(angles_deg)) if angles_deg else [],
    'V_net_mV': volts_angular_mV
})

In [10]:
from scipy.stats import linregress

# --- 1. Calculate Stefan-Boltzmann Constant (sigma) ---
# Using data_temp from earlier
slope_T, intercept_T, r_val_T, _, std_err_T = linregress(data_temp['T4_K4'], data_temp['V_net_mV'])

# Given constants from the lab manual
d = 0.10 # meters (10 cm)
l = 0.01 # meters (1 cm)
sensitivity = 16 # mV/mW

# Calculate sigma in mW/(m^2 K^4) then convert to W/(m^2 K^4)
sigma_exp_mW = slope_T * (4 * d**2) / (np.pi * l**2 * sensitivity)
sigma_exp_W = sigma_exp_mW * 1e-3
sigma_err_W = std_err_T * (4 * d**2) / (np.pi * l**2 * sensitivity) * 1e-3

# Format sigma to standard scientific notation with uncertainty
sigma_formatted = f"{sigma_exp_W:.2e} ± {sigma_err_W:.0e} W/(m^2 K^4)"
sigma_theory = "5.67e-8 W/(m^2 K^4)"

# Create a summary text for the LLM to use in the report
calc_summary = f"""
--- CALCULATED RESULTS (USE THESE IN YOUR TEXT) ---
1. Distance Experiment (1/d^2 vs V):
   - The linear fit shows the relationship between voltage and inverse square of distance.

2. Temperature Experiment (T^4 vs V):
   - Slope = {slope_T:.2e} ± {std_err_T:.0e} mV/K^4
   - Intercept = {intercept_T:.2f} mV
   - R^2 = {r_val_T**2:.4f}

3. Calculated Stefan-Boltzmann Constant (σ):
   - Experimental σ = {sigma_formatted}
   - Theoretical σ = {sigma_theory}
   - (Compare the experimental result to the theoretical value in your conclusion, and discuss sources of error).
"""

In [11]:
todo_prompt = f"""
{instructions}

Here is the theoretical background and procedure for the experiment performed:
---
{sample}
---

I have already calculated the data and constants for you. Here are the results:
{calc_summary}

Please generate the complete lab report in Persian based on the 15-section structure.

CRITICAL INSTRUCTIONS:
1. DO NOT invent numbers. Use the exact calculated values provided in the summary above for Sections 11, 12, and 13.
2. In Section 12 (Graphs), instead of writing the graphs from scratch, output EXACTLY these placeholders:
   - `==== TODO: INSERT GRAPH: Voltage vs 1/d^2 ====`
   - `==== TODO: INSERT GRAPH: Voltage vs T^4 (Stefan-Boltzmann) ====`
3. In Section 10 (Circuit), output: `==== TODO: INSERT CIRCUIT DIAGRAM ====`
4. In Section 13 (Conclusion), explicitly compare the experimental σ with the theoretical σ provided above.
5. Answer the following questions in Section 14:
{questions}

--- RAW DATASET (For reference in Section 11) ---
1. Distance Data:
{data_distance.to_markdown(index=False)}

2. Temperature Data:
{data_temp.to_markdown(index=False)}
"""

In [12]:
# Retrieve API key from environment (loaded via load_dotenv() in Cell 3)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found. Add it to your .env file.")

# --- Web Search Tool ---
# Useful for looking up references, standards, or background info for the lab report
tavily_search = TavilySearch(
    max_results=5,
    topic="general",          # or "news" if you need recent info
    include_answer=True,       # get a short synthesized answer
    include_raw_content=False, # keep payload light; use Extract if you need full text
    tavily_api_key=TAVILY_API_KEY,
)

# --- URL Content Extraction Tool ---
# Useful when the agent finds a promising link and needs the full page text
tavily_extract = TavilyExtract(
    extract_depth="basic",     # "advanced" is slower but parses more content
    tavily_api_key=TAVILY_API_KEY,
)

tools = [tavily_search, tavily_extract]

In [13]:
import re
from langchain.agents.middleware import AgentMiddleware, hook_config
from langchain_core.messages import HumanMessage, AIMessage
from typing import Any

class LabReportValidatorMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 2):
        super().__init__()
        self.max_retries = max_retries

    def _check_significant_figures(self, text: str) -> list[str]:
        issues = []
        # Regex to find: [number] ± [number]
        pattern = r"([+-]?\d+(?:\.\d+)?)\s*[±]\s*([+-]?\d+(?:\.\d+)?)"
        matches = re.finditer(pattern, text)
        
        for match in matches:
            val_str = match.group(1)
            unc_str = match.group(2)
            
            # FIX: Robust significant figure calculation for uncertainty
            try:
                unc_float = float(unc_str)
                if unc_float == 0:
                    continue
                # Convert to scientific notation to easily find sig figs
                unc_sci = f"{unc_float:e}" 
                mantissa = unc_sci.split('e')[0].replace('-', '').replace('.', '')
                unc_sig_figs = len(mantissa.rstrip('0'))
            except ValueError:
                continue
                
            if unc_sig_figs > 2: # Allow up to 2 sig figs for uncertainty, or change to > 1 for strict 1 sig fig
                issues.append(
                    f"Format error: `{match.group(0)}`. The uncertainty (`{unc_str}`) "
                    f"has more than 2 significant figures. It must be rounded (e.g., 0.0124 -> 0.01)."
                )
                continue

            # Check decimal places match
            unc_decimals = len(unc_str.split('.')[1]) if '.' in unc_str else 0
            val_decimals = len(val_str.split('.')[1]) if '.' in val_str else 0

            if val_decimals != unc_decimals:
                issues.append(
                    f"Format error: `{match.group(0)}`. The value (`{val_str}`) has {val_decimals} decimal places, "
                    f"but the uncertainty (`{unc_str}`) has {unc_decimals}. "
                    f"They must match (e.g., 3.457 ± 0.01 -> 3.46 ± 0.01)."
                )
        return issues

    @hook_config(can_jump_to=["model"])
    def after_model(self, state: dict[str, Any], runtime: Any) -> dict[str, Any] | None:
        messages = state["messages"]
        last_msg = messages[-1]

        if not isinstance(last_msg, AIMessage) or last_msg.tool_calls:
            return None

        content = last_msg.content or ""
        retry_count = state.get("_validation_retry", 0)
        issues = []

        # Rule 1: Graphs/Images placeholders
        if "====" not in content:
            issues.append("Response does not contain any `====` placeholder markers for graphs or images.")

        # Rule 2: Sig Figs & Errors
        math_issues = self._check_significant_figures(content)
        issues.extend(math_issues)

        if issues and retry_count < self.max_retries:
            state["_validation_retry"] = retry_count + 1
            correction_text = (
                "Your previous response violates the formatting rules. Specific issues found:\n\n"
                + "\n".join(f"- {i}" for i in issues)
                + "\n\nPlease rewrite the ENTIRE report. Remember:\n"
                  "1. Output `==== TODO: INSERT GRAPH ... ====` for plots and `==== TODO: INSERT IMAGE ... ====` for setup photos/circuits.\n"
                  "2. Ensure ALL uncertainties match the decimal places of the value (e.g., 3.46 ± 0.01)."
            )
            messages.append(HumanMessage(content=correction_text))
            return {"messages": messages, "jump_to": "model"}

        if "_validation_retry" in state:
            del state["_validation_retry"]
        return None

In [14]:
from langchain.agents.middleware import ModelRetryMiddleware, ModelCallLimitMiddleware

agent = create_agent(
    model=ChatDeepSeek(model = "deepseek-chat", temperature=0.7),
    tools=tools,
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),      # Prevent infinite loops
        ModelRetryMiddleware(max_retries=2),         # Network-level retry
        LabReportValidatorMiddleware(max_retries=2), # Content quality check
    ],
)

In [15]:
result = agent.invoke({
    "messages": [{"role": "user", "content": todo_prompt}]
})
print(result["messages"][-1].content)

# گزارشکار آزمایشگاه — تشعشع جسم سیاه

## عنوان آزمایش
بررسی قوانین تشعشع جسم سیاه: قانون استفان–بولتزمن، قانون جابه‌جایی وین و قانون کسینوسی لامبرت

## تاریخ انجام آزمایش
۱۴۰۳/۰۹/۱۲

## تاریخ تحویل گزارشکار
۱۴۰۳/۰۹/۱۹

## نام و نام خانوادگی
[نام و نام خانوادگی دانشجو]

## نام همکار
[نام همکار آزمایشگاه]

## چکیده آزمایش
هدف از این آزمایش، بررسی تجربی قوانین حاکم بر تشعشع جسم سیاه، به‌ویژه قانون استفان–بولتزمن و قانون کسینوسی لامبرت، و همچنین تعیین تجربی ثابت استفان–بولتزمن است. بدین منظور از یک کوره حرارتی به‌عنوان تقریبی از جسم سیاه، یک ترمُوپایل برای اندازه‌گیری توان تشعشعی دریافتی و یک ولت‌متر دقیق برای ثبت ولتاژ خروجی ترمُوپایل استفاده شد. پارامترهای اندازه‌گیری‌شده شامل ولتاژ خروجی ترمُوپایل برحسب فاصله از کوره (برای بررسی قانون عکس مجذور فاصله)، برحسب زاویه تابش (برای بررسی قانون کسینوسی لامبرت) و برحسب دمای مطلق کوره (برای بررسی قانون استفان–بولتزمن) بودند. تحلیل نتایج با برازش خطی داده‌ها و رسم نمودارهای $V$ برحسب $1/d^2$ و $R(T)$ برحسب $T^4$ انجام شد و از شیب خط به‌دست‌آمده ب